In [1]:
import psycopg2
import pandas as pd
import json

# ==========================================
# 1. CẤU HÌNH KẾT NỐI DATABASE
# ==========================================
DB_HOST = "104.155.166.86"
DB_PORT = "5432"
DB_NAME = "car_recsys" # Cập nhật tên DB
DB_USER = "postgres"     # Cập nhật Username

# Lấy mật khẩu an toàn từ Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    DB_PASS = user_secrets.get_secret("DB_PASS")
except Exception as e:
    print("Cảnh báo: Không thể lấy mật khẩu từ Kaggle Secrets. Vui lòng kiểm tra lại thiết lập.")
    DB_PASS = "fallback_password_neu_can"

OUTPUT_FILE = "/kaggle/working/qlora_dataset.jsonl"

# ==========================================
# 2. CÂU LỆNH SQL KÉO & GOM NHÓM DỮ LIỆU
# ==========================================
SQL_QUERY = """
    SELECT 
        v.vehicle_id, 
        v.brand, 
        v.car_model, 
        v.body_type,
        v.year, 
        v.mileage, 
        v.fuel_type, 
        v.transmission, 
        v.exterior_color, 
        v.price,
        m.rating_reliability, 
        m.rating_performance, 
        m.rating_exterior,
        m.rating_interior,
        m.rating_comfort, 
        ARRAY_AGG(img.image_url ORDER BY img.image_order) AS image_urls
    FROM gold.vehicles v
    LEFT JOIN gold.car_models m ON v.car_model = m.car_model
    INNER JOIN gold.vehicle_images img ON v.vehicle_id = img.vehicle_id
    WHERE v.price IS NOT NULL
    GROUP BY 
        v.vehicle_id, v.brand, v.car_model, v.body_type, v.year, v.mileage, 
        v.fuel_type, v.transmission, v.exterior_color, v.price,
        m.rating_reliability, m.rating_performance, m.rating_exterior, m.rating_interior, m.rating_comfort;
"""

print("Đang kết nối Database và truy xuất dữ liệu...")
conn = None
try:
    conn = psycopg2.connect(
        host=DB_HOST, port=DB_PORT, database=DB_NAME, user=DB_USER, password=DB_PASS
    )
    df_clean = pd.read_sql(SQL_QUERY, conn)
    print(f"Truy xuất thành công! Tổng số xe: {len(df_clean)}")
    
except Exception as e:
    print(f"Lỗi kết nối DB: {e}")
finally:
    if conn:
        conn.close()





Đang kết nối Database và truy xuất dữ liệu...


/tmp/ipykernel_16/1294956647.py:61: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_clean = pd.read_sql(SQL_QUERY, conn)


Truy xuất thành công! Tổng số xe: 5337


In [2]:
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
conn = psycopg2.connect(host=DB_HOST, port=DB_PORT, database=DB_NAME, user=DB_USER, password=DB_PASS)
df = pd.read_sql(SQL_QUERY, conn)
conn.close()
print(f"Đã tải {len(df)} dòng dữ liệu thô.\n")

Đã tải 5337 dòng dữ liệu thô.



In [4]:
# 2. EDA 
print("--- 1. BÁO CÁO DỮ LIỆU KHUYẾT THIẾU ---")
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({'Số dòng thiếu': missing_data, 'Tỷ lệ (%)': missing_percent})
print(missing_df[missing_df['Số dòng thiếu'] > 0].sort_values(by='Tỷ lệ (%)', ascending=False))
print("-" * 50)

--- 1. BÁO CÁO DỮ LIỆU KHUYẾT THIẾU ---
                    Số dòng thiếu  Tỷ lệ (%)
rating_performance            293   5.489976
rating_exterior               293   5.489976
rating_interior               293   5.489976
rating_reliability            293   5.489976
rating_comfort                293   5.489976
body_type                      90   1.686341
exterior_color                 40   0.749485
transmission                    5   0.093686
fuel_type                       4   0.074948
year                            1   0.018737
brand                           1   0.018737
car_model                       1   0.018737
--------------------------------------------------


In [5]:
# ==========================================
# 2. LÀM SẠCH DỮ LIỆU (THEO CHIẾN LƯỢC CỦA BẠN)
# ==========================================
print(f"Số dòng ban đầu: {len(df_clean)}")

# Chiến lược 1: Drop các dòng thiếu thông tin cơ bản
drop_cols = ['body_type', 'exterior_color', 'transmission', 'fuel_type', 'year', 'brand', 'car_model', 'mileage']
df_clean = df_clean.dropna(subset=drop_cols)

# Chiến lược 2: Điền Mean theo Hãng xe (Brand) cho các cột Rating
rating_cols = ['rating_reliability', 'rating_performance', 'rating_exterior', 'rating_interior', 'rating_comfort']
for col in rating_cols:
    # Điền mean của từng hãng
    df_clean[col] = df_clean.groupby('brand')[col].transform(lambda x: x.fillna(x.mean()))
    # Nếu có hãng nào toàn bộ đều là NaN (không tính được mean), fallback về mean toàn cục
    df_clean[col] = df_clean[col].fillna(df_clean[col].mean())
    # Làm tròn 1 chữ số thập phân cho đẹp
    df_clean[col] = df_clean[col].round(1)

print(f"Số dòng sau khi làm sạch: {len(df_clean)}")

Số dòng ban đầu: 5337
Số dòng sau khi làm sạch: 5201


In [6]:
# ==========================================
# 3. ĐÓNG GÓI JSONL (100% TIẾNG ANH)
# ==========================================
print("Đang tạo cấu trúc JSONL...")
jsonl_data = []

for index, row in df_clean.iterrows():
    if not isinstance(row['image_urls'], list) or len(row['image_urls']) == 0:
        continue
        
    instruction = (
        f"You are an expert used car appraiser. Please estimate the value of the following vehicle based on its technical specifications and the attached images.\n\n"
        f"--- BASIC SPECIFICATIONS ---\n"
        f"- Brand: {row['brand']}\n"
        f"- Model: {row['car_model']} ({row['body_type']})\n"
        f"- Year: {int(row['year'])}\n"
        f"- Mileage: {int(row['mileage']):,} miles\n"
        f"- Fuel Type: {row['fuel_type']}\n"
        f"- Transmission: {row['transmission']}\n"
        f"- Exterior Color: {row['exterior_color']}\n\n"
        f"--- SYSTEM MODEL RATINGS ---\n"
        f"- Reliability: {row['rating_reliability']}/5.0\n"
        f"- Performance: {row['rating_performance']}/5.0\n"
        f"- Exterior Design: {row['rating_exterior']}/5.0\n"
        f"- Interior Quality: {row['rating_interior']}/5.0\n"
        f"- Comfort: {row['rating_comfort']}/5.0\n\n"
        f"Based on the information above and the provided images, please provide an estimated price (in USD)."
    )
    
    response = (
        f"Based on the overall assessment of the {int(row['year'])} {row['brand']} {row['body_type']}, "
        f"with a current odometer reading of {int(row['mileage']):,} miles and the condition shown in the images, "
        f"a reasonable estimated price for this vehicle is ${float(row['price']):,.0f}."
    )
    
    data_point = {
        "instruction": instruction,
        "images": row['image_urls'][:6],
        "response": response
    }
    jsonl_data.append(data_point)

Đang tạo cấu trúc JSONL...


In [7]:
# ==========================================
# 4. XUẤT FILE JSONL
# ==========================================
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for item in jsonl_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"Hoàn tất Phase 1! Đã lưu file tại: {OUTPUT_FILE}")

Hoàn tất Phase 1! Đã lưu file tại: /kaggle/working/qlora_dataset.jsonl
